# Model Autophagy Disorder (MAD) on chest X-rays — complete pipeline

Everything from the raw MIMIC-CXR-JPG files to the generation-over-generation MAD curve.
Nothing is hidden in external scripts: every preprocessing decision, seed and threshold is in this
notebook so the study is reproducible from the published code alone.

```
raw MIMIC-CXR-JPG
  → patient-disjoint split (train / validate / test)
  → 256px grayscale PNG
  → device filter  (report keywords ∪ CheXpert ∪ NegBio)
  → balance 1:1     (7,047 / class)
  → border autocrop
  → gen-0 GAN
       → generate 100k/class  →  PCA-Mahalanobis 10%  →  herding → 7,047/class
       → gen-1 GAN  →  …  →  gen-N
```

**Set `GEN` in cell 2 and run top to bottom.** `GEN=0` trains the real-data baseline; `GEN≥1` runs one
self-consuming generation. Every stage is idempotent — it checks whether its output exists and skips —
so re-running is safe and a crash costs at most the current stage.

### Design choices a reader will ask about

**Anchored, not fully self-consuming.** Herding matches against a *fixed real anchor* (the same 7,047/class
gen-0 trained on) at every generation. The question this asks is whether filtering toward real data slows
the collapse. A fully self-consuming arm (no real access after gen-0) answers a different question and
would be a separate experiment.

**Fixed evaluation reference.** All generations are scored against the same held-out real images the
generator never trained on, so generation-to-generation numbers are comparable.

**Coverage is the headline, not FID.** MAD manifests as diversity collapse before fidelity loss, and
`copy_rate` rising means later generations are memorising rather than generating.

### Operational rules
1. **One GPU job at a time.** Never run another GPU script alongside a cell here.
2. **Cell 1 first, fresh kernel.** `MIOPEN_FIND_MODE=NORMAL` must be set *before* `stylegan2_ada_cond`
   is imported, or R1 silently runs 34× slower.
3. **Identical training config every generation**, or a difference between generations could be the
   training budget rather than MAD.

## 1. Environment — RUN FIRST, IN A FRESH KERNEL


In [ ]:
import os, sys
os.environ["MIOPEN_FIND_MODE"] = "NORMAL"      # hard assign, NOT setdefault
os.environ.pop("MIOPEN_FIND_ENFORCE", None)
os.environ["WANDB_MODE"] = "disabled"
for p in (r"C:\Users\Tonkid\Downloads", r"C:\Users\Tonkid\Downloads\tstr",
          r"C:\Users\Tonkid\Downloads\tstr\filters"):
    sys.path.insert(0, p)

assert "stylegan2_ada_cond" not in sys.modules, (
    "stylegan2_ada_cond already imported -- MIOPEN_FIND_MODE is locked in.\n"
    "RESTART THE KERNEL and run this cell first.")

import glob, csv, json, time, random, shutil, gc
import numpy as np, pandas as pd, torch
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
print("MIOPEN_FIND_MODE =", os.environ["MIOPEN_FIND_MODE"])
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2. Configuration — `GEN` is the only value you change
`GEN=0` → train the real-data baseline. `GEN=1,2,…` → one self-consuming generation each.

In [ ]:
GEN       = 1                # <<<< generation to BUILD (0 = real-data baseline)

RES       = 256              # image resolution
N_POOL    = 100000           # fakes generated per class before filtering
N_KEEP    = 7047             # kept per class (= the real per-class count after balancing)
MAHA_DROP = 0.10             # PCA-Mahalanobis outlier gate
KIMG      = 1000             # SAME for every generation
SEED      = 0
CLASSES   = ["cardiomegaly", "normal"]
PRIMARY   = "cardiomegaly"   # class the metrics are reported on

# --- source data (raw MIMIC-CXR-JPG) ---
SRC_ROOT  = r"D:\mimic-cardiomegaly"                                  # contains files/p10..p19
MANIFEST  = r"C:\Users\Tonkid\Downloads\tstr\data_cardio_full\manifest_full.csv"
CHEXPERT  = r"C:\Users\Tonkid\Downloads\mimic_cxr_cardiomegaly\mimic-cxr-2.0.0-chexpert.csv.gz"
NEGBIO    = r"C:\Users\Tonkid\Downloads\mimic_cxr_cardiomegaly\mimic-cxr-2.0.0-negbio.csv.gz"
KEYWORDS  = r"C:\Users\Tonkid\Downloads\results\device_filter\report_device_hits.csv"

# --- derived paths ---
DL        = r"C:\Users\Tonkid\Downloads"
FULL_PNG  = rf"{DL}\tstr\data_cardio_full\train"        # step 1 output: all train images at 256px
REAL_TRAIN= rf"{DL}\tstr\data_cardio_final_crop\train"  # gen-0 training set + herding's real anchor
REAL_EVAL = rf"{DL}\tstr\data_cardio_eval_crop"         # held-out reals, never trained on
METRICS   = rf"{DL}\results\mad_recursion.json"

PREV_CKPT = (rf"{DL}\runs\mad_gen{GEN-1}\checkpoints\final.pt" if GEN > 1 else
             rf"{DL}\runs\cardio_final_crop_p06\gen0\checkpoints\final.pt")
POOL_ROOT = rf"{DL}\runs\mad_gen{GEN}\pool"
DATA_ROOT = REAL_TRAIN if GEN == 0 else rf"{DL}\tstr\data_mad_gen{GEN}\train"
RUN_OUT   = (rf"{DL}\runs\cardio_final_crop_p06\gen0" if GEN == 0 else rf"{DL}\runs\mad_gen{GEN}")

print(f"BUILDING gen-{GEN}" + ("  (real-data baseline)" if GEN == 0 else ""))
print(f"  trains on : {DATA_ROOT}")
print(f"  run out   : {RUN_OUT}   ({KIMG} kimg)")
if GEN > 0:
    print(f"  pool from : {PREV_CKPT}")
    print(f"  real anchor for herding: {REAL_TRAIN}")

## 3. Step 0a — source data + patient-disjoint splits
The manifest defines **patient-disjoint** train/validate/test splits (split on `subject_id`, so no patient
appears in two splits). Only `train` is used to build the GAN; `validate` and `test` are reserved for the
downstream classifier probe and are never touched here.

In [ ]:
rows = list(csv.DictReader(open(MANIFEST, newline="")))
def _lab(r): return "cardiomegaly" if r["cardiomegaly_chexpert"] == "1.0" else "normal"

import collections
cnt = collections.Counter((r["split"], _lab(r)) for r in rows)
subj = {s: set() for s in ("train", "validate", "test")}
for r in rows:
    subj[r["split"]].add(r["subject_id"])

print(f"{'split':10s} {'cardiomegaly':>13s} {'normal':>8s} {'total':>8s} {'patients':>9s}")
for s in ("train", "validate", "test"):
    c, n = cnt[(s, "cardiomegaly")], cnt[(s, "normal")]
    print(f"{s:10s} {c:13d} {n:8d} {c+n:8d} {len(subj[s]):9d}")

# the property that matters: no patient in more than one split
for a, b in (("train", "validate"), ("train", "test"), ("validate", "test")):
    ov = subj[a] & subj[b]
    assert not ov, f"PATIENT LEAKAGE between {a} and {b}: {len(ov)}"
print("\npatient-disjoint across all three splits: OK")
assert os.path.isdir(os.path.join(SRC_ROOT, "files")), f"raw images not found at {SRC_ROOT}"

## 4. Step 0b — decode to 256px grayscale PNG
Frontal cardiomegaly studies from the `train` split, converted once. Filenames are
`{subject_id}_{study_id}_{dicom_id}.png` so every image traces back to its manifest row.

In [ ]:
def resolve(relpath):
    rp = relpath.replace("/", os.sep)
    for cand in (rp, rp[len("files" + os.sep):] if rp.lower().startswith("files" + os.sep) else rp):
        p = os.path.join(SRC_ROOT, cand)
        if os.path.isfile(p):
            return p
    return None

have = {c: len(glob.glob(os.path.join(FULL_PNG, c, "*.png"))) for c in CLASSES}
if min(have.values()) > 1000:
    print("already decoded:", have)
else:
    t0, miss, done = time.time(), 0, 0
    for c in CLASSES:
        os.makedirs(os.path.join(FULL_PNG, c), exist_ok=True)
    for r in rows:
        if r["split"] != "train":
            continue
        dst = os.path.join(FULL_PNG, _lab(r),
                           f"{r['subject_id']}_{r['study_id']}_{r['dicom_id']}.png")
        if os.path.exists(dst):
            continue
        src = resolve(r["relpath"])
        if src is None:
            miss += 1; continue
        Image.open(src).convert("L").resize((RES, RES), Image.LANCZOS).save(dst)
        done += 1
        if done % 5000 == 0:
            print(f"  {done} decoded ({(time.time()-t0)/60:.1f} min)", flush=True)
    print(f"decoded {done}, {miss} source files missing")
print("train PNGs:", {c: len(glob.glob(os.path.join(FULL_PNG, c, '*.png'))) for c in CLASSES})

## 5. Step 0c — device filter, balance, border crop → the gen-0 training set

**Device filter.** Support devices (pacemakers, ICDs, lines, sternal wires) correlate with cardiomegaly
(56% vs 47% prevalence), so a model can learn "hardware → cardiomegaly" instead of heart size. A study is
dropped if **any** of three independent signals flags it: report keyword search, the CheXpert labeler, or
the NegBio labeler. The union removes 53.8% of studies. It **cannot** reach device-free — all three read
the report, so unreported hardware survives; this is a stated limitation, not a solved problem.

**Balance.** 1:1 by random subsample (seed 0). `normal` has only 7,047 device-free images, which caps
both classes at **7,047**.

**Border crop.** MIMIC films carry black collimation borders of varying size, a nuisance variable the GAN
would otherwise have to model. `autocrop` trims rows/columns whose mean intensity is below 15% of the
image maximum, then resizes to 256. Validated not to distort the disease signal: cardiothoracic ratio
before vs after gave r=0.988, mean Δ=−0.0007 (n=299), with identical segmentation-failure rates.

In [ ]:
def autocrop(gray_u8, frac=0.15, pad=2):
    """Trim the black collimation border. A border row/column has near-zero MEAN intensity, which is
    robust to bright corner markers (a single bright letter cannot lift a whole line's mean)."""
    a = np.asarray(gray_u8, np.float32)
    rm, cm = a.mean(1), a.mean(0)
    rows_ = np.where(rm > frac * rm.max())[0]
    cols_ = np.where(cm > frac * cm.max())[0]
    if len(rows_) < 8 or len(cols_) < 8:
        return gray_u8                                  # degenerate/too dark -> leave as-is
    r0, r1 = max(rows_[0] - pad, 0), min(rows_[-1] + pad, a.shape[0] - 1)
    c0, c1 = max(cols_[0] - pad, 0), min(cols_[-1] + pad, a.shape[1] - 1)
    return Image.fromarray(a[r0:r1 + 1, c0:c1 + 1].astype(np.uint8))

# ---- the union device-drop set ----
kwset = {str(r["study_id"]) for r in csv.DictReader(open(KEYWORDS, newline=""))}
cx, nb = pd.read_csv(CHEXPERT), pd.read_csv(NEGBIO)
DROP = (kwset
        | set(cx[cx["Support Devices"] == 1.0]["study_id"].astype(str))
        | set(nb[nb["Support Devices"] == 1.0]["study_id"].astype(str)))
print(f"device drop set: {len(DROP)} studies (keyword U CheXpert U NegBio)")

free = {}
for c in CLASSES:
    allp = sorted(glob.glob(os.path.join(FULL_PNG, c, "*.png")))
    free[c] = [p for p in allp if os.path.basename(p).split("_")[1] not in DROP]
    print(f"  {c:14s} {len(allp):6d} -> device-free {len(free[c]):6d} "
          f"({100*(1-len(free[c])/len(allp)):.1f}% dropped)")

n_bal = min(len(v) for v in free.values())
print(f"\nbalancing 1:1 -> {n_bal}/class (capped by the smaller class)")
assert n_bal == N_KEEP, f"balance gives {n_bal}, config expects N_KEEP={N_KEEP}"

rng_bal = random.Random(SEED)
chosen = {}
for c in CLASSES:
    d = os.path.join(REAL_TRAIN, c); os.makedirs(d, exist_ok=True)
    files = free[c] if len(free[c]) == n_bal else rng_bal.sample(free[c], n_bal)
    chosen[c] = {os.path.basename(p) for p in files}
    if len(glob.glob(os.path.join(d, "*.png"))) >= n_bal:
        print(f"  {c:14s} already built"); continue
    for i, src in enumerate(files):
        autocrop(Image.open(src).convert("L")).resize((RES, RES), Image.LANCZOS).save(
            os.path.join(d, os.path.basename(src)))
        if (i + 1) % 2000 == 0:
            print(f"    {c}: {i+1}/{n_bal}", flush=True)
    print(f"  {c:14s} wrote {len(files)} -> {d}")
print("\ngen-0 training set:", {c: len(glob.glob(os.path.join(REAL_TRAIN, c, '*.png'))) for c in CLASSES})

## 6. Step 0d — held-out real evaluation reference
Balancing used only 7,047 of the 16,664 device-free cardiomegaly images, so the remaining **9,617** are
device-free reals the generator **never trained on**. Cropped identically, they become the fixed scoring
reference for every generation — so FID/KID/coverage measure generalisation, not reproduction of the
training images.

*Limitation: image-disjoint, not patient-disjoint (a patient may contribute studies to both sets). The
`copy_rate` metric is what guards against memorisation.*

In [ ]:
ev_d = os.path.join(REAL_EVAL, PRIMARY); os.makedirs(ev_d, exist_ok=True)
held = [p for p in free[PRIMARY] if os.path.basename(p) not in chosen[PRIMARY]]
print(f"held-out device-free {PRIMARY}: {len(held)}")
if len(glob.glob(os.path.join(ev_d, "*.png"))) < len(held):
    for i, src in enumerate(held):
        dst = os.path.join(ev_d, os.path.basename(src))
        if not os.path.exists(dst):
            autocrop(Image.open(src).convert("L")).resize((RES, RES), Image.LANCZOS).save(dst)
        if (i + 1) % 2000 == 0:
            print(f"  {i+1}/{len(held)}", flush=True)
eval_paths = sorted(glob.glob(os.path.join(ev_d, "*.png")))
train_names = {os.path.basename(p) for p in glob.glob(os.path.join(REAL_TRAIN, PRIMARY, '*.png'))}
assert not (train_names & {os.path.basename(p) for p in eval_paths}), "LEAKAGE: train/eval overlap"
print(f"eval reference: {len(eval_paths)} imgs, disjoint from training set: OK")

## 7. Step 0e — probe evaluation sets (validate + test)

The probe measures **downstream utility**: train a cardiomegaly classifier on a generation's data, test it
on real patients. That needs real held-out images with **both** classes.

`train`'s leftovers cannot supply this — balancing consumed every device-free `normal`, so a test set built
from them would have device-free cardiomegaly and device-rich normals, and the classifier could score ~100%
by detecting hardware. `validate` and `test` are untouched and patient-disjoint from `train`, so they can.

Built to **match the training distribution**: device-filtered, autocropped, balanced 1:1.

| split | device-free balanced | role |
|---|---|---|
| validate | 865 / class | iterate here — every generation |
| test | 867 / class | **LOCKED** — run once, at the very end |

*Balanced because the raw splits are 74% cardiomegaly; accuracy would be meaningless and AUROC noisy.*


In [ ]:
PROBE_SPLIT = "validate"      # "validate" while iterating; "test" ONCE, at the very end
PROBE_ROOT  = rf"{DL}\tstr\data_probe"

def build_probe_split(split):
    """Decode -> device filter -> autocrop -> balance 1:1. Idempotent."""
    out = os.path.join(PROBE_ROOT, split)
    have = {c: len(glob.glob(os.path.join(out, c, "*.png"))) for c in CLASSES}
    if min(have.values(), default=0) > 100:
        print(f"{split}: already built {have}"); return out
    keep = {c: [] for c in CLASSES}
    for r in rows:
        if r["split"] != split or str(r["study_id"]) in DROP:      # device filter
            continue
        keep[_lab(r)].append(r)
    n = min(len(v) for v in keep.values())
    print(f"{split}: device-free {[(c, len(keep[c])) for c in CLASSES]} -> balanced {n}/class", flush=True)
    rng_p = random.Random(SEED)
    for c in CLASSES:
        d = os.path.join(out, c); os.makedirs(d, exist_ok=True)
        sel = keep[c] if len(keep[c]) == n else rng_p.sample(keep[c], n)
        for i, r in enumerate(sel):
            dst = os.path.join(d, f"{r['subject_id']}_{r['study_id']}_{r['dicom_id']}.png")
            if os.path.exists(dst):
                continue
            src = resolve(r["relpath"])
            if src is None:
                continue
            autocrop(Image.open(src).convert("L")).resize((RES, RES), Image.LANCZOS).save(dst)
            if (i + 1) % 400 == 0:
                print(f"    {c}: {i+1}/{n}", flush=True)
    return out

for _s in ("validate", "test"):
    build_probe_split(_s)
PROBE_EVAL = os.path.join(PROBE_ROOT, PROBE_SPLIT)
print("\nprobe eval:", PROBE_SPLIT,
      {c: len(glob.glob(os.path.join(PROBE_EVAL, c, "*.png"))) for c in CLASSES})

# patients in the probe eval set must never appear in the GAN's training data
_train_subj = {os.path.basename(p).split("_")[0]
               for c in CLASSES for p in glob.glob(os.path.join(REAL_TRAIN, c, "*.png"))}
_eval_subj  = {os.path.basename(p).split("_")[0]
               for c in CLASSES for p in glob.glob(os.path.join(PROBE_EVAL, c, "*.png"))}
assert not (_train_subj & _eval_subj), "PATIENT LEAKAGE between GAN training and probe eval"
print("patient-disjoint from GAN training data: OK")


## 8. Stage A — generate the pool (skipped when `GEN=0`)
ψ=1.0 and `noise_mode="random"`: the full generated distribution, **not** the truncated view used for
training preview grids. Truncation would hide exactly the tail behaviour MAD is about.

In [ ]:
if GEN == 0:
    print("GEN=0 (real-data baseline) - no pool needed")
else:
    from filter_bakeoff import load_G, gen_pool
    assert os.path.exists(PREV_CKPT), f"missing previous generation: {PREV_CKPT}"
    G, classes, c_dim, gcfg = load_G(PREV_CKPT)
    for cls in CLASSES:
        d = os.path.join(POOL_ROOT, cls); os.makedirs(d, exist_ok=True)
        n_have = len(glob.glob(os.path.join(d, "*.png")))
        if n_have >= N_POOL:
            print(f"{cls}: {n_have} already on disk - skip"); continue
        ci, t0, B = classes.index(cls), time.time(), 250
        print(f"{cls}: generating {n_have} -> {N_POOL} ...", flush=True)
        for start in range(n_have, N_POOL, B):
            k = min(B, N_POOL - start)
            imgs = gen_pool(G, gcfg, c_dim, ci, k, batch=50, seed=1000 + start)
            for j in range(k):
                Image.fromarray(imgs[j].permute(1, 2, 0).numpy()).convert("L").save(
                    os.path.join(d, f"{start+j:06d}.png"))
            if (start // B) % 40 == 0:
                print(f"    {start+k}/{N_POOL}  ({(time.time()-t0)/60:.1f} min)", flush=True)
        print(f"  done ({(time.time()-t0)/60:.1f} min)", flush=True)
    del G; torch.cuda.empty_cache(); gc.collect()

## 9. Stage B — filter: PCA-Mahalanobis 10% → herding (skipped when `GEN=0`)

**Gate first, selector last.** Herding optimises the set *jointly* (each pick depends on those already
chosen), so deleting images afterwards would break its solution and miss the target n.

**PCA-Mahalanobis (10%)** drops the fakes farthest from the real centre. In a controlled comparison it was
metric-neutral (ΔKID inside one standard deviation) but the images it removes are ~30–40% structurally
broken — which distributional metrics cannot see, and which compounds across generations.

**Herding** greedily minimises MMD to the real anchor using KID's own polynomial kernel. Against a random
control at matched n it gave coverage 0.524→0.731, KID −86%, FID −47% on held-out reals.

In [ ]:
if GEN == 0:
    print("GEN=0 - training set is the real data, no filtering")
else:
    import features as FE, gates as Gt, herding as H
    os.makedirs(RUN_OUT, exist_ok=True)
    for cls in CLASSES:
        out_d = os.path.join(DATA_ROOT, cls)
        if len(glob.glob(os.path.join(out_d, "*.png"))) >= N_KEEP:
            print(f"{cls}: training set already built - skip"); continue
        pool_paths = sorted(glob.glob(os.path.join(POOL_ROOT, cls, "*.png")))[:N_POOL]
        anchor     = sorted(glob.glob(os.path.join(REAL_TRAIN, cls, "*.png")))
        print(f"\n[{cls}] pool {len(pool_paths)} | real anchor {len(anchor)}", flush=True)
        pf = FE.extract_features(pool_paths, backbone="inception")
        sf = FE.extract_features(anchor,     backbone="inception")

        keep, info = Gt.q_maha(pf, sf, drop_frac=MAHA_DROP, seed=SEED)
        surv = np.where(keep)[0]
        print(f"  PCA-Mahalanobis {int(MAHA_DROP*100)}%: dropped {(~keep).sum()} -> {len(surv)} survive",
              flush=True)

        t0 = time.time()
        loc, diag = H.greedy_mmd_select(pf[surv], sf, N_KEEP, kernel="poly",
                                        kernel_kwargs={"degree": 3}, device="cuda", log_every=3000)
        idx = surv[np.asarray(loc)]
        print(f"  herding: kept {len(idx)}  ({time.time()-t0:.0f}s)", flush=True)

        os.makedirs(out_d, exist_ok=True)
        for j in np.sort(idx):
            src = pool_paths[j]; dst = os.path.join(out_d, os.path.basename(src))
            if not os.path.exists(dst):
                try: os.link(src, dst)
                except OSError: shutil.copy2(src, dst)
        np.save(os.path.join(RUN_OUT, f"kept_{cls}.npy"), idx)
        print(f"  exported -> {out_d}", flush=True)
        del pf, sf; gc.collect(); torch.cuda.empty_cache()
print("\ntraining set:", {c: len(glob.glob(os.path.join(DATA_ROOT, c, '*.png'))) for c in CLASSES})

## 10. Stage C — visual check before training
100 images per class from what will actually be trained on. Junk here propagates to every later
generation, and no distributional metric will warn you about a few percent of broken frames.

In [ ]:
for cls in CLASSES:
    paths = sorted(glob.glob(os.path.join(DATA_ROOT, cls, "*.png")))
    pick = [paths[i] for i in np.random.default_rng(5).choice(len(paths), 100, replace=False)]
    CELL, GAP = 200, 4; W = 10 * CELL + 11 * GAP
    sheet = Image.new("L", (W, W), 32)
    for i, p in enumerate(pick):
        r_, c_ = divmod(i, 10)
        sheet.paste(Image.open(p).convert("L").resize((CELL, CELL)),
                    (GAP + c_ * (CELL + GAP), GAP + r_ * (CELL + GAP)))
    dst = rf"{DL}\MADGEN{GEN}_{cls}_100.png"
    sheet.save(dst); print("saved", dst)

## 11. Stage D — train gen-N (GPU, ~13 h)

Config is **identical for every generation**, so any change across generations is MAD and not the budget.

`p_max=0.6` caps ADA's augmentation probability. Uncapped, ADA drove p to 1.0 by ~kimg 100 and the
generator learned the *augmented* distribution — in a controlled A/B at kimg 600 that cost FID +5.8,
KID +0.008 (5× the error bar) and coverage −0.07. `allow_flip_rot=False` because heart laterality is
diagnostic and chest films are never sideways.

Auto-resumes from `latest.pt` if present.

In [ ]:
from stylegan2_ada_cond.train import Config, train

os.makedirs(RUN_OUT, exist_ok=True)
latest = os.path.join(RUN_OUT, "checkpoints", "latest.pt")
resume = latest if os.path.exists(latest) else None
print(("RESUMING from " + latest) if resume else f"FRESH gen-{GEN} training", flush=True)

cfg = Config(
    data=DATA_ROOT, out=RUN_OUT, conditional=True, xflip=False, resolution=RES,
    batch=16, kimg=KIMG, lr=0.0015, r1_gamma=1.0, ema_kimg=10.0,
    p_init=0.5, ada_kimg=100, ada_target=0.6, allow_flip_rot=False, p_max=0.6,
    sample_every_kimg=20, ckpt_every_kimg=100, fid_every_kimg=0,
    amp=True, r1_cudnn=None, workers=0, seed=SEED, resume=resume,
)
train(cfg)
print(f"=== gen-{GEN} reached {KIMG} kimg ===")

## 12. Stage E — score gen-N against the fixed held-out reals
FID/KID (Inception-v3), coverage/density/recall (Naeem 2020, Kykäänniemi 2019), Vendi, and `copy_rate`.
Appended to `results/mad_recursion.json`.

*`copy_rate` baseline is 0.05 by construction (it is the 5th percentile of real–real nearest-neighbour
distance); a value well above that means the model is reproducing individual patients.*

In [ ]:
import cardio_metrics as M, features as FE
from filter_bakeoff import load_G, gen_pool
from prdc_vendi import coverage_density, recall, vendi_score

CK = os.path.join(RUN_OUT, "checkpoints", "final.pt")
assert os.path.exists(CK), "train this generation first"
real_imgs = FE.load_paths_uint8(eval_paths)
print(f"scoring gen-{GEN} vs {len(real_imgs)} held-out reals", flush=True)

G, classes, c_dim, gcfg = load_G(CK)
fake = gen_pool(G, gcfg, c_dim, classes.index(PRIMARY), 5000, batch=50, seed=7)
del G; torch.cuda.empty_cache()
fk = M.fid_kid(real_imgs, fake)

fo = M._new_fid()
rf_, ff_ = M._features(fo, real_imgs).numpy(), M._features(fo, fake).numpy()
del fo, real_imgs, fake; torch.cuda.empty_cache()
cd, rc = coverage_density(rf_, ff_, k=5), recall(rf_, ff_, k=5)
vi = np.random.default_rng(SEED).choice(len(ff_), 2000, replace=False)
vd = vendi_score(ff_[vi], kernel="cosine")
R = torch.as_tensor(rf_, device="cuda"); F_ = torch.as_tensor(ff_, device="cuda")
drr = torch.cdist(R, R); drr.fill_diagonal_(float("inf"))
tau = torch.quantile(drr.min(1).values.float(), 0.05)
cr = float((torch.cdist(F_, R).min(1).values < tau).float().mean())
del R, F_, drr; torch.cuda.empty_cache()

row = {"gen": GEN, "kimg": KIMG, "n_fake": 5000, "fid": fk["fid"], "kid": fk["kid"],
       "kid_std": fk["kid_std"], "coverage": cd["coverage"], "density": cd["density"],
       "recall": rc["recall"], "vendi": vd["vendi"], "copy_rate": round(cr, 4)}
print(f"gen-{GEN}: FID {fk['fid']:.2f}  KID {fk['kid']:+.5f}  coverage {cd['coverage']:.4f}  "
      f"vendi {vd['vendi']:.1f}  copy_rate {cr:.3f}")

hist = json.load(open(METRICS)) if os.path.exists(METRICS) else {"rows": []}
hist["rows"] = [r for r in hist["rows"] if r["gen"] != GEN] + [row]
hist["rows"].sort(key=lambda r: r["gen"])
os.makedirs(os.path.dirname(METRICS), exist_ok=True)
json.dump(hist, open(METRICS, "w"), indent=2)
print("saved ->", METRICS)

## 13. Stage E2 — cardiothoracic ratio: is the *disease* still there? (GPU, ~15 min)

Every other metric is indirect. FID/KID measure distance in an ImageNet feature space; AUROC measures what
a classifier can exploit. **CTR is the clinical definition of cardiomegaly** — maximum heart width divided
by maximum thoracic width, measured from an anatomical segmentation (XRV PSPNet). A film is conventionally
called cardiomegalic at CTR > 0.5.

Three numbers per generation:

| | what it catches |
|---|---|
| **mean CTR per class** | drift of the disease signal away from the real distribution |
| **CTR gap** (cardiomegaly − normal) | **the key one** — if it shrinks, the conditional GAN is losing the class distinction entirely |
| **segmentation failure rate** | anatomy degrading to the point the segmenter cannot find a heart or lungs |

The gap is the sharpest signal: images can stay individually plausible while the two classes quietly
converge, and no distributional metric names that. The real-data gap is measured once and reused as the
reference for every generation.

*CTR is invariant to the crop's aspect-ratio stretch (it is a ratio of two horizontal widths), which is why
border-cropping was safe — verified r=0.988 before vs after.*


In [ ]:
import cardio_metrics as M
N_CTR   = 1000                              # images per class (PSPNet at 512px is the slow part)
CTR_REF = rf"{DL}\results\ctr_real_reference.json"

def ctr_stats(imgs):
    v = np.asarray(M.ctr_values(imgs))
    ok = ~np.isnan(v)
    return {"mean": float(v[ok].mean()) if ok.any() else float("nan"),
            "std": float(v[ok].std()) if ok.any() else float("nan"),
            "fail": float((~ok).mean())}

# ---- real reference, measured once and reused for every generation ----
if os.path.exists(CTR_REF):
    ref = json.load(open(CTR_REF))
    print("real CTR reference (cached):", {k: round(v["mean"], 4) for k, v in ref.items()})
else:
    ref = {}
    for cls in CLASSES:
        src = (sorted(glob.glob(os.path.join(REAL_EVAL, cls, "*.png"))) if cls == PRIMARY
               else sorted(glob.glob(os.path.join(REAL_TRAIN, cls, "*.png"))))
        pick = [src[i] for i in np.random.default_rng(SEED).choice(len(src), min(N_CTR, len(src)),
                                                                   replace=False)]
        ref[cls] = ctr_stats(FE.load_paths_uint8(pick))
        print(f"  real {cls:14s} CTR {ref[cls]['mean']:.4f} +/-{ref[cls]['std']:.4f}  "
              f"(seg fail {ref[cls]['fail']*100:.1f}%)", flush=True)
    json.dump(ref, open(CTR_REF, "w"), indent=2)
ref_gap = ref["cardiomegaly"]["mean"] - ref["normal"]["mean"]
print(f"  real CTR gap (cardiomegaly - normal): {ref_gap:+.4f}\n")

# ---- this generation ----
G, classes, c_dim, gcfg = load_G(CK)
gen_ctr = {}
for cls in CLASSES:
    imgs = gen_pool(G, gcfg, c_dim, classes.index(cls), N_CTR, batch=50, seed=21)
    gen_ctr[cls] = ctr_stats(imgs)
    d = gen_ctr[cls]["mean"] - ref[cls]["mean"]
    print(f"  gen-{GEN} {cls:14s} CTR {gen_ctr[cls]['mean']:.4f} +/-{gen_ctr[cls]['std']:.4f}  "
          f"(vs real {d:+.4f}, seg fail {gen_ctr[cls]['fail']*100:.1f}%)", flush=True)
    del imgs; gc.collect(); torch.cuda.empty_cache()
del G; torch.cuda.empty_cache()

gap = gen_ctr["cardiomegaly"]["mean"] - gen_ctr["normal"]["mean"]
print(f"\n  gen-{GEN} CTR gap: {gap:+.4f}   (real {ref_gap:+.4f}, retained "
      f"{100*gap/ref_gap:.0f}% of the class separation)")

hist = json.load(open(METRICS)) if os.path.exists(METRICS) else {"rows": []}
row = next((r for r in hist["rows"] if r["gen"] == GEN), {"gen": GEN})
row.update({"ctr_cardiomegaly": round(gen_ctr["cardiomegaly"]["mean"], 4),
            "ctr_normal": round(gen_ctr["normal"]["mean"], 4),
            "ctr_gap": round(gap, 4), "ctr_gap_real": round(ref_gap, 4),
            "ctr_gap_retained": round(gap / ref_gap, 4),
            "ctr_seg_fail": round(max(gen_ctr[c]["fail"] for c in CLASSES), 4)})
hist["rows"] = [r for r in hist["rows"] if r["gen"] != GEN] + [row]
hist["rows"].sort(key=lambda r: r["gen"])
json.dump(hist, open(METRICS, "w"), indent=2)
print("saved ->", METRICS)


## 14. Stage F — probe score for THIS generation (GPU, ~1 h)

**The test that actually matters.** Distribution metrics say the data *looks* different; the probe says
whether it is still *useful*. They can disagree, and that disagreement is a result, not a bug.

The probe trains on exactly what this generation feeds forward — `DATA_ROOT`, the 7,047/class that will
train the next generation (for `GEN=0`, the real data, giving the **TRTR ceiling**).

Two probes, deliberately different in kind:

| probe | setup | why |
|---|---|---|
| **DenseNet-121** | ImageNet init, **fully fine-tuned**, 20 epochs | medical-imaging CNN baseline |
| **DINOv2 ViT-B/14** | **frozen** backbone + linear head | modern self-supervised foundation model |

If both agree that utility degrades, the finding is much harder to attribute to one architecture's quirk.

**The recipe is frozen.** Do not tune epochs, LR or augmentation between generations — the variable under
study is the training *data*, not the classifier. Metric: **AUROC** (primary), rare-class recall (secondary).


In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.metrics import roc_auc_score, recall_score
from sklearn.linear_model import LogisticRegression

PROBE_EPOCHS, PROBE_BATCH, PROBE_LR = 20, 32, 1e-4

class FolderDS(Dataset):
    """Class-folder dataset; label 1 = cardiomegaly."""
    def __init__(self, root, train_aug):
        self.items = [(p, 1 if c == "cardiomegaly" else 0)
                      for c in CLASSES for p in sorted(glob.glob(os.path.join(root, c, "*.png")))]
        ops = [T.Grayscale(num_output_channels=3), T.Resize((RES, RES))]
        if train_aug:
            ops += [T.RandomHorizontalFlip(0.0)]          # no flip: heart laterality is diagnostic
        ops += [T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
        self.tf = T.Compose(ops)
    def __len__(self):  return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p)), y

train_ds = FolderDS(DATA_ROOT,  train_aug=True)
eval_ds  = FolderDS(PROBE_EVAL, train_aug=False)
print(f"probe train {len(train_ds)} (from gen-{GEN} data) | eval {len(eval_ds)} ({PROBE_SPLIT})", flush=True)

def evaluate(probs, ys):
    pred = (np.asarray(probs) > 0.5).astype(int)
    return {"auroc": float(roc_auc_score(ys, probs)),
            "recall_cardiomegaly": float(recall_score(ys, pred, pos_label=1, zero_division=0)),
            "recall_normal": float(recall_score(ys, pred, pos_label=0, zero_division=0))}

# ---------------- probe 1: DenseNet-121, fully fine-tuned ----------------
# MIOpen's BATCHNORM kernels fail on gfx1101 (RuntimeError: miopenStatusUnknownError).
# StyleGAN2 uses no BatchNorm, so this project never hit that path before. The remedy is
# the one train.py applies to the broken im2col conv kernel -- route the layer through
# native ATen -- but it must be scoped to BatchNorm ALONE. Disabling the backend around
# the whole model also strips MIOpen from all 121 conv layers, which is the slow
# native-conv path (the one that costs R1 34x), and the probe never finishes.
class NativeBN(nn.BatchNorm2d):
    """BatchNorm2d that bypasses MIOpen; conv layers keep their MIOpen kernels."""
    def forward(self, x):
        with torch.backends.cudnn.flags(enabled=False):
            return super().forward(x)

def swap_bn(mod):
    for name, child in mod.named_children():
        if isinstance(child, nn.BatchNorm2d):
            bn = NativeBN(child.num_features, child.eps, child.momentum,
                          child.affine, child.track_running_stats)
            bn.load_state_dict(child.state_dict())
            setattr(mod, name, bn)
        else:
            swap_bn(child)
    return mod

t0 = time.time()
m = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
m.classifier = nn.Linear(m.classifier.in_features, 2)
m = swap_bn(m).cuda().train()
print(f"  BatchNorm layers routed to native ATen: "
      f"{sum(isinstance(x, NativeBN) for x in m.modules())}", flush=True)
opt = torch.optim.Adam(m.parameters(), lr=PROBE_LR)
lossf = nn.CrossEntropyLoss()
dl = DataLoader(train_ds, batch_size=PROBE_BATCH, shuffle=True, num_workers=0,
                drop_last=True)   # BatchNorm needs >1 sample per batch
for ep in range(PROBE_EPOCHS):
    tot = 0.0
    for xb, yb in dl:
        opt.zero_grad(set_to_none=True)
        loss = lossf(m(xb.cuda()), yb.cuda())
        loss.backward(); opt.step(); tot += float(loss)
    print(f"  densenet epoch {ep+1}/{PROBE_EPOCHS}  loss {tot/len(dl):.4f}  "
          f"({(time.time()-t0)/60:.1f} min elapsed)", flush=True)

m.eval(); probs, ys = [], []
with torch.no_grad():
    for xb, yb in DataLoader(eval_ds, batch_size=PROBE_BATCH):
        probs += torch.softmax(m(xb.cuda()), 1)[:, 1].cpu().tolist(); ys += yb.tolist()
dn = evaluate(probs, ys)
print(f"  DenseNet-121  AUROC {dn['auroc']:.4f}  recall_cardio {dn['recall_cardiomegaly']:.3f}", flush=True)
del m, opt; torch.cuda.empty_cache(); gc.collect()

# ---------------- probe 2: DINOv2 ViT-B/14, frozen + linear head ----------------
import features as FE
tr_paths = [p for c in CLASSES for p in sorted(glob.glob(os.path.join(DATA_ROOT, c, "*.png")))]
tr_y     = [1 if "cardiomegaly" in os.path.dirname(p) else 0 for p in tr_paths]
ev_paths = [p for c in CLASSES for p in sorted(glob.glob(os.path.join(PROBE_EVAL, c, "*.png")))]
ev_y     = [1 if "cardiomegaly" in os.path.dirname(p) else 0 for p in ev_paths]
Xtr = FE.extract_features(tr_paths, backbone="dinov2")
Xev = FE.extract_features(ev_paths, backbone="dinov2")
clf = LogisticRegression(max_iter=2000, C=1.0).fit(Xtr, tr_y)
dv = evaluate(clf.predict_proba(Xev)[:, 1], ev_y)
print(f"  DINOv2 linear AUROC {dv['auroc']:.4f}  recall_cardio {dv['recall_cardiomegaly']:.3f}", flush=True)

# ---------------- record into this generation's row ----------------
hist = json.load(open(METRICS)) if os.path.exists(METRICS) else {"rows": []}
row = next((r for r in hist["rows"] if r["gen"] == GEN), {"gen": GEN})
row.update({"probe_split": PROBE_SPLIT, "probe_n_train": len(train_ds), "probe_n_eval": len(eval_ds),
            "auroc_densenet": round(dn["auroc"], 4), "recall_densenet": round(dn["recall_cardiomegaly"], 4),
            "auroc_dinov2": round(dv["auroc"], 4), "recall_dinov2": round(dv["recall_cardiomegaly"], 4)})
hist["rows"] = [r for r in hist["rows"] if r["gen"] != GEN] + [row]
hist["rows"].sort(key=lambda r: r["gen"])
json.dump(hist, open(METRICS, "w"), indent=2)
print("saved ->", METRICS)


## 15. The MAD curve — every generation so far
**Read coverage first.** Model autophagy shows up as diversity collapse before fidelity loss, so a rising
FID is the late symptom, not the early warning. A climbing `copy_rate` means later generations are
memorising rather than generating.

In [ ]:
hist = json.load(open(METRICS))
R = hist["rows"]
print(f"{'gen':>4} {'FID':>8} {'KID':>10} {'coverage':>9} {'vendi':>7} {'copy':>6} "
      f"{'CTRgap':>7} {'AUROC-dn':>9} {'AUROC-dv':>9}")
print("-" * 84)
for r in R:
    f = lambda k, w, p: (f"{r[k]:{w}.{p}f}" if r.get(k) is not None else " " * w)
    print(f"{r['gen']:>4} {f('fid',8,2)} {f('kid',10,5)} {f('coverage',9,4)} "
          f"{f('vendi',7,1)} {f('copy_rate',6,3)} {f('ctr_gap',7,4)} "
          f"{f('auroc_densenet',9,4)} {f('auroc_dinov2',9,4)}")

if len(R) > 1:
    g0 = R[0]
    print(f"\nchange vs gen-{g0['gen']}:")
    for r in R[1:]:
        parts = []
        for k, lab, p in (("coverage", "coverage", 4), ("kid", "KID", 5), ("vendi", "vendi", 1),
                          ("copy_rate", "copy", 3), ("ctr_gap", "CTRgap", 4),
                          ("auroc_densenet", "AUROC-dn", 4),
                          ("auroc_dinov2", "AUROC-dv", 4)):
            if r.get(k) is not None and g0.get(k) is not None:
                parts.append(f"{lab} {r[k]-g0[k]:+.{p}f}")
        print(f"  gen-{r['gen']}: " + "   ".join(parts))

try:
    import matplotlib.pyplot as plt
    gens = [r["gen"] for r in R]
    panels = [("coverage", "coverage (higher=better)", None),
              ("kid", "KID (lower=better)", None),
              ("copy_rate", "copy_rate (memorisation)", 0.05),
              ("ctr_gap", "CTR gap (class separation)", None),
              ("auroc_densenet", "AUROC DenseNet-121", 0.5),
              ("auroc_dinov2", "AUROC DINOv2", 0.5)]
    fig, ax = plt.subplots(1, len(panels), figsize=(4 * len(panels), 3.4))
    for a, (key, ttl, ref) in zip(ax, panels):
        xs = [r["gen"] for r in R if r.get(key) is not None]
        ys = [r[key] for r in R if r.get(key) is not None]
        a.plot(xs, ys, "o-")
        if ref is not None:
            a.axhline(ref, ls="--", c="r", lw=1)
        a.set_title(ttl); a.set_xlabel("generation"); a.set_xticks(gens); a.grid(alpha=.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print("(matplotlib not installed - table only)")
